# 7-to-1 |Y⟩ State Distillation (Steane Code)

Reference: `processing/Steane_distillation.png` Figure 46(c)

**Circuit structure:**
- W1, W2, W3: injected |Y⟩ (noisy)
- W4: fault-tolerant |+⟩
- A: ancilla |Y⟩ (reused per π/4 rotation)
- 4 sequential Z product measurements
- End: MX on W1-W3 (post-select), S†+MX on W4 (output)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.qec_code.surface_code.unrotated import (
    UnrotatedSurfaceCode,
    UnrotatedSurfaceCodeExtractionBlock,
    UnrotatedMultiPatchCoupler,
    UnrotatedSurfaceCodeLogicalOpSet,
)
from src.ir.qec_system import QECSystem
from src.ir.builder import CircuitBuilder
from src.ir.tracker import SyndromeTracker
import numpy as np

## Step 1: Patch Layout

All patches rotated π/2 (Z boundary faces corridor). All-side-patch layout.

```
W1(-2,0)    W2(10,0)
W3(-2,8)    W4(10,8)
A(-2,16)
```

In [ ]:
d = 3
rounds = d

# Create patches
patch_layout = {
    'W1': (-2, 0),   # left top
    'W2': (10, 0),   # right top
    'W3': (-2, 8),   # left mid
    'W4': (10, 8),   # right mid
    'A':  (-2, 16),  # left bottom (ancilla)
}

system = QECSystem()
for name, offset in patch_layout.items():
    p = UnrotatedSurfaceCode(distance=d)
    p.rotate_coords(np.pi/2)
    p.reset_rotation_angle()
    system.add_patch(p, name=name, offset=offset)

print("Patch bounds:")
for name in patch_layout:
    print(f"  {name}: {system.patches[name][0]._get_bounds()}")

print(f"\nSystem: {system.num_qubits} qubits, {system.num_logicals} logicals")

## Step 2: Simplified Test — Single Z Product Measurement

Before the full 4-measurement Steane circuit, test ONE Z product measurement
with sequential coupler reuse. This verifies the infrastructure works.

Test: Z₃ measurement on {W1, W2, W3} → deactivate → remove → done.

In [ ]:
# Single Z product measurement test (using if_detector=False for now)
# This tests the remove_coupler → re-register cycle

coupler_name = 'meas_1'
center = 6.0

# Setup tracker + builder
tracker = SyndromeTracker(num_qubits=system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system, if_detector=False)
builder.write_coordinates()

# Init all code patches in Z (noiseless for testing)
init_dict = {q: 'Z' for q in system.data_indices}
builder.initialize(init_dict=init_dict, n=system.num_qubits)

# SE rounds (stabilize before measurement)
se = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

# Register coupler: ZZZ on W1, W2, W3
system.register_coupler(UnrotatedMultiPatchCoupler(),
    patch_names=['W1', 'W2', 'W3'], name=coupler_name,
    path_axis='vertical', center_axis=center)

# Activate + init coupler data in X + SE
builder.activate_coupler(coupler_name)
cp = system.coupler_patches[coupler_name]
cd = [system.local_to_global_map[coupler_name][q] for q in cp.data_indices]
ci = {q: 'X' for q in cd}
builder.initialize(init_dict=ci, n=system.num_qubits)

se2 = UnrotatedSurfaceCodeExtractionBlock(system)
builder.apply_syndrome_extraction(circuit_chunk=se2.circuit, rounds=rounds)

# Deactivate + remove
builder.deactivate_coupler(coupler_name)
system.remove_coupler(coupler_name)

# Final readout (all code patches in Z, skip coupler data)
measure_dict = {q: 'Z' for q in system.data_indices}
builder.apply_data_readout(final_measurements=measure_dict)

circuit = builder.circuit
print(f"Single measurement test: {circuit.num_qubits} qubits")
circuit.diagram("detslice-with-ops-svg")

## Step 3: Two Sequential Z Product Measurements

Test the remove_coupler → re-register cycle:
1. ZZZ on {W1, W2, W3}
2. Remove coupler
3. ZZ on {W1, W4} (different subset, same corridor space)
4. Remove coupler
5. Final readout

In [ ]:
# Two sequential Z product measurements with coupler reuse

# Fresh system (re-use patch layout from Step 1)
system2 = QECSystem()
for name, offset in patch_layout.items():
    p = UnrotatedSurfaceCode(distance=d)
    p.rotate_coords(np.pi/2)
    p.reset_rotation_angle()
    system2.add_patch(p, name=name, offset=offset)

center = 6.0
tracker2 = SyndromeTracker(num_qubits=system2.num_qubits, expected_num_logicals=system2.num_logicals)
builder2 = CircuitBuilder(tracker=tracker2, system_config=system2, if_detector=False)
builder2.write_coordinates()

# Init all patches in Z
init2 = {q: 'Z' for q in system2.data_indices}
builder2.initialize(init_dict=init2, n=system2.num_qubits)

se = UnrotatedSurfaceCodeExtractionBlock(system2)
builder2.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)

def do_z_measurement(sys, builder, tracker, patch_names, coupler_name, center_axis):
    """Perform one Z product measurement cycle."""
    # Register
    sys.register_coupler(UnrotatedMultiPatchCoupler(),
        patch_names=patch_names, name=coupler_name,
        path_axis='vertical', center_axis=center_axis)
    
    # Expand tracker if needed
    n = sys.num_qubits
    if n > tracker.num_qubits:
        tracker.expand(n - tracker.num_qubits)
        builder.write_coordinates()
    
    # Activate + init coupler data + SE
    builder.activate_coupler(coupler_name)
    cp = sys.coupler_patches[coupler_name]
    cd = [sys.local_to_global_map[coupler_name][q] for q in cp.data_indices]
    ci = {q: 'X' for q in cd}
    builder.initialize(init_dict=ci, n=n)
    
    se = UnrotatedSurfaceCodeExtractionBlock(sys)
    builder.apply_syndrome_extraction(circuit_chunk=se.circuit, rounds=rounds)
    
    # Deactivate + remove
    builder.deactivate_coupler(coupler_name)
    sys.remove_coupler(coupler_name)
    
    print(f"  {coupler_name}: ZZZ on {patch_names} — done")

# Measurement 1: ZZZ on W1, W2, W3
print("Sequential measurements:")
do_z_measurement(system2, builder2, tracker2, ['W1', 'W2', 'W3'], 'meas_1', center)

# Measurement 2: ZZ on W1, W4 (different subset, same corridor)
do_z_measurement(system2, builder2, tracker2, ['W1', 'W4'], 'meas_2', center)

# Final readout
m2 = {q: 'Z' for q in system2.data_indices}
builder2.apply_data_readout(final_measurements=m2)

print(f"\nCircuit: {builder2.circuit.num_qubits} qubits")
print("Two sequential Z measurements compiled successfully!")
builder2.circuit.diagram("detslice-with-ops-svg")